In [11]:
from transformers import pipeline
import pandas as pd
import os
import numpy as np

### Global Constants; can be changed ###
LABELS = ["economic relief", "reopening", "jobs", "housing", "vaccines",
          "testing", "positive cases", "healthcare professionals", 
          "healthcare infrastructure", "other", "research", "food"]
MULTI_LABEL = True
CHARACTER_NUMBER = 100 #number of characers to slice from each press release
WORKING_DIRECTORY = "/Users/connorrust/Library/CloudStorage/Box-Box/Covid Policies/"
INPUT_DATA = "Data/05_combine_all_states.csv"
OUTPUT_PATH = "Analysis/Testing/Results/06_burnham_all_states.csv"
#####################################
os.chdir(WORKING_DIRECTORY)
data = pd.read_csv(INPUT_DATA)

# extracting text from data
text = data.pop("Text").str.slice(0,CHARACTER_NUMBER)
lst = text.to_list()[1:5]

hypothesis_template = "This text is about {}"
classes_verbalized = LABELS

zeroshot_classifier = pipeline("zero-shot-classification", 
                               model="mlburnham/Political_DEBATE_DeBERTa_large_v1.1", 
                               device = "mps")  # change the model identifier here

output = zeroshot_classifier(lst, classes_verbalized, 
                             hypothesis_template=hypothesis_template, 
                             multi_label=MULTI_LABEL)

clean_output = []

for dct in output:
    nd = {}
    nd["sequence"] = dct["sequence"]
    for idx, label in enumerate(dct["labels"]):
        nd[label] = dct["scores"][idx]
    clean_output.append(nd)

df3 = pd.DataFrame(clean_output)




Loading weights: 100%|██████████| 394/394 [00:00<00:00, 6487.20it/s]


In [12]:
df3

,sequence,economic relief,reopening,positive cases,other,testing,food,healthcare infrastructure,housing,jobs,vaccines,healthcare professionals,research
0,SACRAMENTO – Governor Gavin Newsom will today ...,9.995472e-01,2.637096e-05,2.088211e-05,0.000005,2.408934e-06,9.377753e-07,6.468366e-07,4.892846e-07,4.647899e-07,3.324896e-07,1.985342e-07,1.582566e-07
1,Information regarding 2021 Governor’s Office m...,1.615965e-07,2.106630e-06,1.662418e-07,0.000005,1.602867e-07,1.573480e-07,1.701422e-07,1.617073e-07,2.534268e-07,1.649744e-07,1.525641e-07,1.733946e-07
2,SACRAMENTO – California is cleaning up encampm...,1.921027e-07,9.090522e-07,2.073435e-07,0.000051,1.765373e-07,1.567079e-07,1.617040e-07,2.888418e-05,1.638895e-07,1.695750e-07,1.718489e-07,1.609895e-07
3,SACRAMENTO – Governor Gavin Newsom and state h...,1.675590e-07,2.418915e-04,4.916950e-07,0.000016,9.999943e-01,1.823021e-07,4.189932e-03,1.710777e-07,1.696694e-07,5.780639e-07,3.865493e-06,2.059711e-07


In [4]:
df1 = pd.DataFrame(entailment[1])
df1.pivot(index = "sequence", columns = "labels", values = "scores")

labels,economic relief,food,healthcare infrastructure,healthcare professionals,housing,jobs,other,positive cases,reopening,research,testing,vaccines
sequence,,,,,,,,,,,,
Information regarding 2021 Governor’s Office media credentials: Working members of the media who reg,1.615965e-07,1.573480e-07,1.701422e-07,1.525641e-07,1.617073e-07,2.534268e-07,0.000005,1.662418e-07,0.000002,1.733946e-07,1.602867e-07,1.649744e-07


In [21]:
df

,sequence,labels,scores
0,SACRAMENTO – Governor Gavin Newsom will today ...,"[economic relief, reopening, positive cases, o...","[0.9995471835136414, 2.637095531099476e-05, 2...."
1,Information regarding 2021 Governor’s Office m...,"[other, reopening, jobs, research, healthcare ...","[4.717694991995813e-06, 2.1066302906547207e-06..."
2,SACRAMENTO – California is cleaning up encampm...,"[other, housing, reopening, positive cases, ec...","[5.148530544829555e-05, 2.8884178391308524e-05..."
3,SACRAMENTO – Governor Gavin Newsom and state h...,"[testing, healthcare infrastructure, reopening...","[0.9999943375587463, 0.004189932253211737, 0.0..."


In [16]:
dict = {}
dict["rice"] = "chicken"


KeyError: 'chicken'

In [ ]:


### Global Constants; can be changed ###
LABELS = ["economic relief", "reopening", "jobs", "housing", "vaccines",
          "testing", "positive cases", "healthcare professionals", 
          "healthcare infrastructure", "other", "research", "food"]
MULTI_LABEL = True
CHARACTER_NUMBER = 100 #number of characers to slice from each press release
WORKING_DIRECTORY = "/Users/connorrust/Library/CloudStorage/Box-Box/Covid Policies/"
INPUT_DATA = "Data/05_combine_all_states.csv"
OUTPUT_PATH = "Analysis/Testing/Results/06_burnham_all_states.csv"
#####################################
os.chdir(WORKING_DIRECTORY)
data = pd.read_csv(INPUT_DATA)

# defining normalization function
def normalize(matrix, axis=-1):
    """ Takes a numpy 2D array of topic probabilities and normalizes them. 
    (This applies L1 not L2 normalization)
    Args: 
        matrix(numpy array)
        axis(int): axis to normalize across, 1: rows; 0: columns

    Returns:
        2D array with rows/columns normalized

    """
    return matrix / np.sum(matrix, axis=axis, keepdims = True)

# extracting text from data
text = data.pop("Text").str.slice(0,CHARACTER_NUMBER)
lst = text.to_list()[1:5]

ent_template = "This text is about {}"
dis_template = "This text is not about {}"

classes_verbalized = LABELS

zeroshot_classifier = pipeline("zero-shot-classification", 
                               model="mlburnham/Political_DEBATE_DeBERTa_large_v1.1", 
                               device = "mps")  # change the model identifier here

entailment = zeroshot_classifier(lst, classes_verbalized, 
                             hypothesis_template=ent_template, 
                             multi_label=MULTI_LABEL)

disentailment = zeroshot_classifier(lst, classes_verbalized, 
                             hypothesis_template=dis_template, 
                             multi_label=MULTI_LABEL)

clean_output = []

for edct, ddct in zip(entailment, disentailment):
    nd = {}
    nd["sequence"] = edct["sequence"]

    if edct["sequence"] != ddct["sequence"]:
        raise ValueError("entailment and disentailment sequences do not match")
    
    for idx, label in enumerate(edct["labels"]):
        nd[label + "_ent"] = edct["scores"][idx]
    for idx, label in enumerate(ddct["labels"]):
        nd[label + "_dis"] = ddct["scores"][idx]
        
    clean_output.append(nd)

df = pd.DataFrame(clean_output)

Loading weights: 100%|██████████| 394/394 [00:00<00:00, 9788.57it/s]


In [7]:
df

,sequence,economic relief_ent,reopening_ent,positive cases_ent,other_ent,testing_ent,food_ent,healthcare infrastructure_ent,housing_ent,jobs_ent,...,vaccines_dis,housing_dis,healthcare infrastructure_dis,jobs_dis,food_dis,reopening_dis,testing_dis,other_dis,positive cases_dis,economic relief_dis
0,SACRAMENTO – Governor Gavin Newsom will today ...,9.995472e-01,2.637096e-05,2.088211e-05,0.000005,2.408934e-06,9.377753e-07,6.468366e-07,4.892846e-07,4.647899e-07,...,1.0,1.000000,0.999999,0.999999,0.999996,0.999985,0.999982,0.999970,0.840110,0.000002
1,Information regarding 2021 Governor’s Office m...,1.615965e-07,2.106630e-06,1.662418e-07,0.000005,1.602867e-07,1.573480e-07,1.701422e-07,1.617073e-07,2.534268e-07,...,1.0,1.000000,1.000000,0.999999,1.000000,1.000000,1.000000,0.999997,1.000000,1.000000
2,SACRAMENTO – California is cleaning up encampm...,1.921027e-07,9.090522e-07,2.073435e-07,0.000051,1.765373e-07,1.567079e-07,1.617040e-07,2.888418e-05,1.638895e-07,...,1.0,0.999533,1.000000,1.000000,1.000000,1.000000,1.000000,0.999994,1.000000,1.000000
3,SACRAMENTO – Governor Gavin Newsom and state h...,1.675590e-07,2.418915e-04,4.916950e-07,0.000016,9.999943e-01,1.823021e-07,4.189932e-03,1.710777e-07,1.696694e-07,...,1.0,1.000000,0.001496,1.000000,1.000000,0.999916,0.000003,0.999998,0.999999,1.000000
